# How to Perform Trajectory and Gene Gradient Field Analysis

This notebook shows the trajectory and gene-gradient workflow with the public `flowmap` package. The code is deliberately plain: FlowMap does the analysis, and Matplotlib is used only for minimal visual checks.


## Setup

Set cache directories before importing `flowmap`/UMAP. This avoids numba and matplotlib cache issues when the notebook is run from a fresh environment.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from flowmap.geometry import FixedPointAnalyzer, LagrangianPathOptimizer, GeneGradientAnalyzer
from flowmap.plot import plot_velocity_stream
from flowmap.utils import load_dataset

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "tutorial").exists() and (PROJECT_ROOT.parent / "tutorial").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent


/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Data and Fitted FlowMap Object

The tutorial data is a FlowMap wrapper around the fitted embedder. It keeps the pieces we need here: the embedder, cell labels, gene names, and gene-level expression/velocity matrices. The file is not bundled with the repository; download or generate it and place it at `tutorial/larry_flowmap_data.joblib`.


In [2]:
DATA_PATH = PROJECT_ROOT / "tutorial" / "larry_flowmap_data.joblib"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing Larry FlowMap tutorial data: {DATA_PATH}")

data = load_dataset(DATA_PATH)
emb = data.embedder
labels = data.obs["state_info"].astype(str).to_numpy()

print(f"X_emb: {emb.X_emb.shape}, V_emb: {emb.V_emb.shape}")
print(f"genes: {len(data.var_names)}")
print(f"labels: {pd.Series(labels).value_counts().head(10).to_dict()}")


X_emb: (49302, 2), V_emb: (49302, 2)
genes: 2000
labels: {'Undifferentiated': 23770, 'Neutrophil': 8555, 'Monocyte': 8165, 'Baso': 5514, 'Mast': 1414, 'Meg': 1035, 'Erythroid': 365, 'Lymphoid': 203, 'Eos': 168, 'Ccr7_DC': 64}


## 1. Fixed Points

A fixed point is a place where the fitted velocity field is close to zero. FlowMap searches for these points by evaluating the velocity field on a `grid_resolution × grid_resolution` grid.

Because single-cell velocity fields are noisy, FlowMap first smooths the grid-speed field (`speed_smoothing=1.2`) and keeps only low-speed candidate regions (`speed_quantile_threshold=0.06`).


In [ ]:
# FlowMap core analysis: identify candidate locations where velocity is near zero.
fpa = FixedPointAnalyzer(emb)
fixed_points = fpa.identify_fixed_points(
    grid_resolution=90,
    speed_smoothing=1.2,
    speed_quantile_threshold=0.06,
)

pd.DataFrame([
    {
        "fp": i,
        "x": fp["position"][0],
        "y": fp["position"][1],
        "type": fp["type"],
        "speed": fp.get("speed", np.nan),
    }
    for i, fp in enumerate(fixed_points)
])


In [ ]:
# Visualization only: show fixed points on the FlowMap velocity stream plot.
fig, ax = plt.subplots(figsize=(6, 6))
plot_velocity_stream(emb.X_emb, spline=emb.spline_vf, scatter_color=labels, ax=ax)

for fp in fixed_points:
    ax.scatter(*fp["position"], marker="x", s=80, color="red")

plt.show()


## 2. One Least-Action Path

A least-action path gives a best transition path connecting two points under the fitted velocity field. The only required choices here are the `start` and `end` positions.

The graph initialization can be built in embedding space or original space. We use `distance_mode="orig"` here so the first path proposal respects distances in the original feature space.


In [ ]:
# FlowMap core analysis: choose a start and end point, then compute one path.
start = fixed_points[0]["position"]
end = np.array([2.3, -7.6])

lap = LagrangianPathOptimizer(emb, D=1.0, lam=1e-4)
path_result = lap.fit_path(
    start=start,
    end=end,
    distance_mode="orig",
)

path = path_result["path_refined"]


In [ ]:
# Visualization only: show the path on the same velocity field.
fig, ax = plt.subplots(figsize=(6, 6))
plot_velocity_stream(
    emb.X_emb,
    spline=emb.spline_vf,
    scatter_color=labels,
    scatter_alpha=0.06,
    ax=ax,
)

for fp in fixed_points:
    ax.scatter(*fp["position"], marker="x", s=80, color="red")

ax.scatter(start[0], start[1], s=70, color="#56B4E9", edgecolor="white", linewidth=0.8, zorder=6)
ax.scatter(end[0], end[1], s=70, color="#56B4E9", edgecolor="white", linewidth=0.8, zorder=6)
ax.plot(path[:, 0], path[:, 1], color="#CC79A7", linewidth=4, alpha=0.95, zorder=7)
plt.show()


## 3. Cells Near the Path

For the gene analysis, we only use cells close to this path. This keeps the gradient calculation local to the transition we care about.


In [ ]:
# Analysis setup: find cells close to the path in embedding coordinates.
scale = np.mean(np.ptp(emb.X_emb, axis=0))
radius = 0.05 * scale

dist_to_path = np.linalg.norm(
    emb.X_emb[:, None, :] - path[None, :, :],
    axis=2,
).min(axis=1)
neighbor_indices = np.where(dist_to_path <= radius)[0]

print(f"{len(neighbor_indices):,} cells near the path")


In [ ]:
# Visualization only: highlight the local path neighborhood.
fig, ax = plt.subplots(figsize=(6, 6))
plot_velocity_stream(
    emb.X_emb,
    spline=emb.spline_vf,
    scatter_color=labels,
    scatter_alpha=0.04,
    ax=ax,
)
ax.scatter(
    emb.X_emb[neighbor_indices, 0],
    emb.X_emb[neighbor_indices, 1],
    s=16,
    color="#F0E442",
    alpha=0.55,
    linewidth=0,
    zorder=5,
)
ax.scatter(start[0], start[1], s=70, color="#56B4E9", edgecolor="white", linewidth=0.8, zorder=6)
ax.scatter(end[0], end[1], s=70, color="#56B4E9", edgecolor="white", linewidth=0.8, zorder=6)
ax.plot(path[:, 0], path[:, 1], color="#CC79A7", linewidth=4.5, alpha=0.95, zorder=7)
plt.show()


## 4. Full Gene-Level Spline Fit

FlowMap needs a gene-level spline to ask how each gene changes across the embedding. We fit this spline on all Larry cells here. The next step only computes Jacobians for path-neighbor cells, so the expensive gradient tensor is still local.


In [ ]:
X_gene = data.layers["spliced"]
V_gene = data.layers["velocity"]

# FlowMap core analysis: fit gene-level expression and velocity splines.
emb.n_spline_points = None
emb.gene_names = data.var_names
emb.fit_gene_level_splines(X=X_gene, V=V_gene, dof_gene=50, dof_vf_gene=50)


## 5. Gene Gradients Along the Path

For each gene, FlowMap computes the gene-expression gradient and compares it to the local velocity direction. The analyzer computes gene Jacobians during initialization, so we pass `cell_indices=neighbor_indices` instead of computing a full `49,302 × 2,000 × 2` Jacobian tensor.


In [ ]:
# FlowMap core analysis: compute path-local gene-gradient directions.
analyzer = GeneGradientAnalyzer(
    emb,
    cell_indices=neighbor_indices,
    verbose=True,
)

gene_indices = np.arange(len(emb.gene_names))
out = analyzer.compute_relative_gradients(
    neighbor_indices,
    gene_indices,
    weight="magnitude",
)

gradient_df = pd.DataFrame({
    "gene_idx": out["gene_indices"],
    "gene": emb.gene_names[out["gene_indices"]],
    "relative_angle": out["angles"],
    "magnitude": out["magnitudes"],
}).sort_values("magnitude", ascending=False)

gradient_df.head(20)


## 6. Radial Plot

The gene angles are relative to the average velocity direction along the path. For plotting, we add that average direction back. Color shows the average directional derivative `J_gene @ v` along the path; the percentile cap below is only a display scale for the color map.


In [ ]:
# Visualization only: radial summary of all path-local gene gradients.
selected_genes = gradient_df["gene_idx"].to_numpy()
relative_angles = gradient_df["relative_angle"].to_numpy()
magnitudes = gradient_df["magnitude"].to_numpy()

v_mean = emb.V_emb[neighbor_indices].mean(axis=0)
base_angle = np.arctan2(v_mean[1], v_mean[0])
scatter_angles = base_angle + relative_angles

# Per-gene color: average directional derivative along local velocity.
J = analyzer.jacobians[:, selected_genes, :]
V = emb.V_emb[analyzer.cell_indices]
vals = np.einsum("cgd,cd->cg", J, V).mean(axis=0)
cap = np.percentile(np.abs(vals), 99)
cap = cap if cap > 0 else 1.0

genes_to_label = ["Fth1", "Srgn", "H2-Aa", "S100a9"]
label_idx = []
for gene in genes_to_label:
    matches = np.where(emb.gene_names == gene)[0]
    if len(matches) == 0:
        continue
    loc = np.where(selected_genes == matches[0])[0]
    if len(loc) > 0:
        label_idx.append(loc[0])

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": "polar"})
ax.set_theta_zero_location("E")
ax.set_theta_direction(1)

ax.plot(
    [0, base_angle],
    [0, min(np.max(magnitudes), np.linalg.norm(v_mean))],
    color="0.2",
    lw=2,
)

ax.scatter(
    scatter_angles,
    magnitudes,
    s=18,
    c=vals,
    cmap="coolwarm",
    vmin=-cap,
    vmax=cap,
    alpha=0.7,
)

for idx in label_idx:
    ax.text(
        scatter_angles[idx],
        magnitudes[idx],
        emb.gene_names[selected_genes[idx]],
        fontsize=8,
    )

plt.show()


## 7. Expression Heatmap

Order the neighboring cells by their closest point on the path, then plot expression for top gradient genes. We threshold weak expression before z-scoring so sparse, low-level expression does not wash out the heatmap.


In [ ]:
from scipy.ndimage import gaussian_filter1d

# Prepare a path-ordered expression matrix.
heatmap_genes = gradient_df.head(15)["gene_idx"].to_numpy()
nearest_path_point = np.linalg.norm(
    emb.X_emb[neighbor_indices, None, :] - path[None, :, :],
    axis=2,
).argmin(axis=1)
cell_order = np.argsort(nearest_path_point)

expr = X_gene[neighbor_indices][:, heatmap_genes][cell_order]

# Threshold weak values gene-by-gene, then z-score for display.
threshold = np.quantile(expr, 0.60, axis=0, keepdims=True)
expr = np.where(expr >= threshold, expr, 0.0)
expr = np.log1p(expr)
expr = (expr - expr.mean(axis=0, keepdims=True)) / (expr.std(axis=0, keepdims=True) + 1e-8)
expr = np.clip(expr, -2.5, 2.5)
expr = gaussian_filter1d(expr, sigma=2.0, axis=0)


In [ ]:
# Visualization only: heatmap of path-ordered expression.
fig, ax = plt.subplots(figsize=(6, 4))
ax.imshow(expr.T, aspect="auto")
ax.set_yticks(np.arange(len(heatmap_genes)))
ax.set_yticklabels(emb.gene_names[heatmap_genes])
plt.show()


## 8. Gene Gradient Fields

Here are simple gradient stream plots for four genes. Replace the gene list with your final choices after checking the radial plot.


In [ ]:
# Visualization only: gene-gradient quivers for a few genes in monocyte cells.
genes_to_plot = ["Fth1", "Srgn", "H2-Aa", "S100a9"]
genes_to_plot = [gene for gene in genes_to_plot if gene in emb.gene_names]
gene_indices_to_plot = [np.where(emb.gene_names == gene)[0][0] for gene in genes_to_plot]

monocyte_indices = np.where(labels == "Monocyte")[0]
rng = np.random.default_rng(1)
plot_indices = rng.choice(
    monocyte_indices,
    size=min(500, len(monocyte_indices)),
    replace=False,
)

X_plot = emb.X_emb[plot_indices]
expr_plot = emb.spline_gene.predict(X_plot)
jac_plot = emb.spline_gene.compute_jacobians(X_plot)

fig = plt.figure(figsize=(10, 7))
gs = fig.add_gridspec(2, 3, width_ratios=[1.15, 1, 1], wspace=0.25, hspace=0.25)

ax0 = fig.add_subplot(gs[:, 0])
plot_velocity_stream(
    emb.X_emb,
    spline=emb.spline_vf,
    scatter_color=labels,
    scatter_alpha=0.05,
    ax=ax0,
)
ax0.scatter(
    emb.X_emb[monocyte_indices, 0],
    emb.X_emb[monocyte_indices, 1],
    s=7,
    color="#E69F00",
    alpha=0.35,
    linewidth=0,
)
ax0.set_title("Monocyte cells")

axes = [fig.add_subplot(gs[i, j]) for i in range(2) for j in [1, 2]]
for ax, gene, gene_idx in zip(axes, genes_to_plot, gene_indices_to_plot):
    grad = jac_plot[:, gene_idx, :]
    expr = expr_plot[:, gene_idx]

    ax.scatter(X_plot[:, 0], X_plot[:, 1], c=expr, s=8, alpha=0.45)
    ax.quiver(
        X_plot[:, 0], X_plot[:, 1],
        grad[:, 0], grad[:, 1],
        angles="xy",
        scale_units="xy",
        scale=12,
        width=0.004,
        alpha=0.85,
    )
    ax.set_title(gene)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)

plt.show()


## Summary

The core FlowMap calls are: fixed-point detection, least-action path computation, full gene-level spline fitting, and path-local gene-gradient analysis. The remaining cells are intentionally plain Matplotlib visual checks.
